# Игнат рояль

In [2]:
import pandas as pd
import os

In [7]:
# Список имен файлов (можно также использовать os.listdir() для автоматизации)
files = [
    'input/2020_выгрузка.xlsx', 
    'input/2021_выгрузка.xlsx', 
    'input/2022_выгрузка.xlsx', 
    'input/2023_выгрузка.xlsx', 
    'input/2024_выгрузка.xlsx'
]

# Создаем пустые списки для накопления данных
dtp_frames = []
participant_frames = []
vehicle_frames = []

def clean_columns(df):
    """Приводит названия колонок к snake_case и убирает пробелы"""
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')
        .str.replace('/', '_')
        .str.replace('.', '')
    )
    return df

# Цикл по файлам
for file in files:
    print(f"Обработка файла: {file}...")
    
    # 1. Читаем листы
    # Используем None в sheet_name, чтобы прочитать все листы разом (вернет словарь)
    print(f"Обработка файла: {file}...")
    
    # Создаем объект ExcelFile, чтобы сначала «заглянуть» внутрь файла
    xls = pd.ExcelFile(file)
    all_sheets = xls.sheet_names
    
    # Создаем словарь маппинга: {название_в_нижнем_регистре: реальное_название}
    # Например: {'дтп': 'ДТП', 'участники': 'Участники'}
    sheet_map = {s.lower(): s for s in all_sheets}
    
    # Безопасно извлекаем реальные названия листов
    # Если листа нет, вернется ошибка (что логично), но регистр теперь не важен
    dtp_sheet = sheet_map['дтп']
    parts_sheet = sheet_map['участники']
    ts_sheet = sheet_map['тс']
    
    # Теперь читаем конкретные листы, используя их настоящие имена из файла
    df_dtp = clean_columns(pd.read_excel(xls, sheet_name=dtp_sheet))
    df_parts = clean_columns(pd.read_excel(xls, sheet_name=parts_sheet))
    df_vehicles = clean_columns(pd.read_excel(xls, sheet_name=ts_sheet))
    
    # 3. Добавляем год источника (полезно для отчета по качеству)
    year = file.split('_')[0]
    df_dtp['source_year'] = year
    
    # 4. Складываем в списки
    dtp_frames.append(df_dtp)
    participant_frames.append(df_parts)
    vehicle_frames.append(df_vehicles)

# Финальная склейка (pd.concat соберет всё в один большой DataFrame)
fact_dtp = pd.concat(dtp_frames, ignore_index=True)
fact_participant = pd.concat(participant_frames, ignore_index=True)
fact_vehicle = pd.concat(vehicle_frames, ignore_index=True)

# Приведение ID к формату из ТЗ
# В листе ДТП главная колонка 'id' должна стать 'dtp_id' для связи
fact_dtp = fact_dtp.rename(columns={'id': 'dtp_id'})

# В листе Участники 'id' это 'participant_id'
fact_participant = fact_participant.rename(columns={'id': 'participant_id'})

# В листе ТС 'id' это 'vehicle_id'
fact_vehicle = fact_vehicle.rename(columns={'id': 'vehicle_id'})

print("Сборка завершена успешно!")

Обработка файла: input/2020_выгрузка.xlsx...
Обработка файла: input/2020_выгрузка.xlsx...
Обработка файла: input/2021_выгрузка.xlsx...
Обработка файла: input/2021_выгрузка.xlsx...
Обработка файла: input/2022_выгрузка.xlsx...
Обработка файла: input/2022_выгрузка.xlsx...
Обработка файла: input/2023_выгрузка.xlsx...
Обработка файла: input/2023_выгрузка.xlsx...
Обработка файла: input/2024_выгрузка.xlsx...
Обработка файла: input/2024_выгрузка.xlsx...
Сборка завершена успешно!


In [15]:
# Оставляем только тех участников, чей dtp_id есть в итоговой таблице ДТП
fact_participant = fact_participant[fact_participant['dtp_id'].isin(fact_dtp['dtp_id'])]

# Аналогично для ТС
fact_vehicle = fact_vehicle[fact_vehicle['dtp_id'].isin(fact_dtp['dtp_id'])]

In [19]:
# Один общий словарь для всех таблиц (rename проигнорирует отсутствующие ключи)
mapping = {
    # Блок: География и Окружение
    'вид_дтп': 'type_name',
    'округ': 'district_name',
    'place_path': 'place_full_path',
    'region_code': 'region_code',
    'okato_code': 'okato_code',
    
    # Блок: Участники
    'тип_тс': 'transport_category_name',  # из листа Участники
    'принадлежность_тс': 'owner_type_name',
    'возраст': 'person_age',
    'дата_дтп': 'dtp_date_ref', # технический дубль даты на листе участников
    'дата': 'date_ref',
    
    # Блок: ТС (Транспортные средства)
    'year': 'manufacture_year', # чтобы не путать с source_year
    'owner_organization': 'owner_org_name',
    'transport_type_name': 'transport_type_name',
    
    # Дополнительные поля для чистоты (если захочешь привести всё к идеалу)
    'transp_amount': 'vehicles_count',
    'suffer_amount': 'injured_count',
    'loss_amount': 'dead_count',
    'suffer_child_amount': 'injured_children_count',
    'loss_child_amount': 'dead_children_count',
    'suffer_child_amount_16': 'injured_children_under_16_count',
    'loss_child_amount_16': 'dead_children_under_16_count'
}

# Применяем ко всем таблицам
fact_dtp.rename(columns=mapping, inplace=True)
fact_participant.rename(columns=mapping, inplace=True)
fact_vehicle.rename(columns=mapping, inplace=True)

Проверка норм ни норм

In [20]:
def get_quality_report(df, name):
    print(f"=== Отчет по таблице: {name} ===")
    
    # 1. Пропуски
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    if not nulls.empty:
        print(f"Пропуски по колонкам:\n{nulls}\n")
    else:
        print("Пропусков не обнаружено.\n")
        
    # 2. Полные дубликаты
    duplicates_count = df.duplicated().sum()
    print(f"Полных дубликатов строк: {duplicates_count}\n")
    
    return {"name": name, "nulls": nulls, "dupes": duplicates_count}

# Запускаем базовый аудит
report_dtp = get_quality_report(fact_dtp, "fact_dtp")
report_part = get_quality_report(fact_participant, "fact_participant")
report_veh = get_quality_report(fact_vehicle, "fact_vehicle")

# 3. Логические аномалии
print("=== Логические аномалии ===")

# Проверка возраста
if 'person_age' in fact_participant.columns:
    age_anomalies = fact_participant[(fact_participant['person_age'] < 0) | (fact_participant['person_age'] > 110)]
    print(f"Аномалий возраста ( <0 или >110): {len(age_anomalies)}")

# Проверка дат (должны быть в диапазоне 2020-2024)
fact_dtp['moment_date'] = pd.to_datetime(fact_dtp['moment_date'], dayfirst=True)
date_anomalies = fact_dtp[(fact_dtp['moment_date'].dt.year < 2020) | (fact_dtp['moment_date'].dt.year > 2024)]
print(f"Записей вне диапазона 2020-2024: {len(date_anomalies)}")

# 4. Конфликты связей (Integrity Check)
print("\n=== Проверка связей (Referential Integrity) ===")

# Участники без ДТП
orphan_parts = fact_participant[~fact_participant['dtp_id'].isin(fact_dtp['dtp_id'])]
print(f"Участников без родительской записи в ДТП: {len(orphan_parts)}")

# ТС без ДТП
orphan_veh = fact_vehicle[~fact_vehicle['dtp_id'].isin(fact_dtp['dtp_id'])]
print(f"ТС без родительской записи в ДТП: {len(orphan_veh)}")

=== Отчет по таблице: fact_dtp ===
Пропуски по колонкам:
scheme_code                 24561
place_full_path                 8
district_name                   8
road_name                   35087
road_loc                    35015
street_name                 10282
house_num                   10213
cut_code                       11
cut_pr_code                    11
traffic_lane_amount            11
dtp_traffic_lane               12
traffic_area_width             11
wayside_width                  12
sidewalk_width                 12
center_mall_width              12
cmall_type_code                12
srf_code                       11
tr_area_state_code             11
light_type_code                11
rate_code                      11
road_constructions_here         8
road_constructions_there        8
meteo_clouds                    8
road_drawbacks                  8
motion_influences               8
region_code                 41131
okato_code                  41131
road_code                

In [ ]:
# Сохранение таблиц
# Parquet сохраняет типы данных, CSV — универсален для БД
tables = {
    'fact_dtp': fact_dtp,
    'fact_participant': fact_participant,
    'fact_vehicle': fact_vehicle
}

for name, df in tables.items():
    # df.to_parquet(f"{name}.parquet", index=False)
    df.to_csv(f"output/{name}.csv", index=False, encoding='utf-8-sig')

In [24]:
# Создание сводного словаря полей
# Используем ранее созданный словарь mapping
data_dict = []
for old_name, new_name in mapping.items():
    # Определяем, в какой таблице искать тип данных
    dtype = "unknown"
    for df in tables.values():
        if new_name in df.columns:
            dtype = str(df[new_name].dtype)
            break
    
    data_dict.append({
        "Исходное поле": old_name,
        "Целевое поле (Unified)": new_name,
        "Тип данных": dtype,
        "Описание": "Приведено к стандарту PEP8 / Snake_case"
    })

df_dict = pd.DataFrame(data_dict)
df_dict.to_excel("data_dictionary.xlsx", index=False)

print("Все файлы сохранены: 3 CSV, 3 Parquet и Data Dictionary.")

Все файлы сохранены: 3 CSV, 3 Parquet и Data Dictionary.


In [28]:
fact_dtp.columns

Index(['dtp_id', 'moment_date', 'moment_time', 'type_code', 'type_name',
       'vehicles_count', 'dead_count', 'dead_children_count', 'injured_count',
       'injured_children_count', 'scheme_code', 'place_latitude',
       'place_longitude', 'place_full_path', 'district_name', 'road_name',
       'road_loc', 'street_name', 'house_num', 'cut_code', 'cut_pr_code',
       'traffic_lane_amount', 'dtp_traffic_lane', 'traffic_area_width',
       'wayside_width', 'sidewalk_width', 'center_mall_width',
       'cmall_type_code', 'srf_code', 'tr_area_state_code', 'light_type_code',
       'rate_code', 'road_constructions_here', 'road_constructions_there',
       'meteo_clouds', 'road_drawbacks', 'motion_influences',
       'injured_children_under_16_count', 'dead_children_under_16_count',
       'area_id', 'district_id', 'source_year', 'region_code', 'okato_code',
       'road_code', 'road_sign_code', 'road_type_code'],
      dtype='str')